In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.io as sio
import netCDF4 as nc
import src.daaa as daaa

In [ ]:
#import the PACE dataset
PACE = nc.Dataset("../../../Documents/PACE/PACE_OCI.20250623T102951.L2.SFREFL.V3_1.nc")

In [ ]:
#read the subsection of the data that you are interested in
cube = PACE.groups['geophysical_data']['rhos'][300:600,400:900]

In [ ]:
#reshape to 2-d cube
dat = cube.data
dat = dat.reshape(-1,122)
#deal with any 0s in the data
dat[dat<0]=0
#set minimum sensitivity level
dat += 1e-3

In [ ]:
#normalize the data - this helps with some issues regarding the mixing of bright and dark objects
datx = (dat[:,:].T/np.sqrt(np.sum(dat[:,:]**2, axis=-1))).T

In [ ]:
#initialize a deterministic annealing archetype analysis unmixing
pacenet = daaa.DAAA(n_components=3, initial_regularization=1, time_constant=100, delta=0, epochs=10)

In [ ]:
#train it
pacenet.PPA=False # turn on to exchange AA for PPA
pacenet._initialize(datx.T) 
#pacenet.verbose=False turn this on to stop the output spam
pacenet.train(datx.T)

In [ ]:
#display the spectra that are found
plt.plot(pacenet.W)

In [ ]:
#display the spectra spatially
plt.imshow(np.rot90(pacenet.H[2].reshape(300,500),4)[::-1,::1])
plt.colorbar()

In [ ]:
#Normalization prevents inversion, here we run without normalization, but weigh all the pixels evenly
#We also turn on PPA so that endmembers must have the spectra of a pixel in the data
pacenetII = daaa.DAAA(n_components=10, initial_regularization=1, time_constant=10, delta=0, epochs=10)
pacenetII.PPA=True
pacenetII.use_weights=True
pacenetII._initialize(dat.T)
pacenetII.train(dat.T)

In [ ]:
#it can be helpful to display the un-normalized data on a log-scale
plt.semilogy(pacenetII.W)

In [ ]:
plt.imshow(pacenetII.H[0].reshape(300,500))
plt.colorbar()

In [ ]:
#To avoid local minima better (but use much more time) use simulated annealing instead of deterministic annealing
pacenetIII = daaa.DAAA(n_components=10, initial_regularization=1, time_constant=100, delta=0, epochs=10)
pacenetIII.PPA=True
pacenetIII.use_weights=False
pacenetIII._initialize(datx.T)
#this is the function for simulated-annealing based training
pacenetIII.SA_update_series(datx, gamma=0)

In [ ]:
plt.imshow(pacenetIII.H[0].reshape(300,500))
plt.colorbar()

In [ ]:
plt.semilogy(pacenetIII.W, label=np.arange(10))
plt.legend()